# Interim Subsample Tools
Using python's ability to implment arbitrary macro expantions.

NB: This is not the right location for this function (nside of /inspector/) but it probides a better environment for prototyping.

In [16]:
# Prep the environment

project="measurement-lab"

import os
import collections
import pandas as pd

# Set project explicitly in the environment to suppress some warnings.
os.environ["GOOGLE_CLOUD_PROJECT"] = project


In [27]:
# Depends on: pip install --upgrade google-cloud-bigquery
from google.cloud import bigquery

def run_query(query, **kwargs):
    global project, rawQuery, numberedQuery
    client = bigquery.Client(project)

    # publish rawQuery and numberedQuery to help with diagnsis
    rawQuery=query.format(**kwargs).format(**kwargs)  # Allow nested args
    numberedQuery=[]
    for n, l in enumerate(rawQuery.splitlines(), start=1):
        numberedQuery.append(f"{n} {l}".format(n, l))
    numberedQuery = "\n".join(numberedQuery)
        
    job = client.query(rawQuery)

    results = collections.defaultdict(list)
    for row in job.result():
        for key in row.keys():
            results[key].append(row.get(key))

    return pd.DataFrame(results)

In [37]:
# Atomically update one table by partition
# 
updateQ="""
# DML
CREATE OR REPLACE TABLE `{dst}_tmp` 
  PARTITION BY date
  {clusterBy}
  {tableOpts}
AS (
    SELECT *, {topColumns}
    FROM `{src}`  -- ('{replaceDay}')  Function form
    WHERE date = '{replaceDay}'
        AND FARM_FINGERPRINT({uuidCol}) & 0xF =  0
);

BEGIN TRANSACTION;
    -- delete the entire partition 
    DELETE FROM `{dst}` WHERE date = '{replaceDay}'; -- Normally a noop
    
    -- insert the new data into the same partition in mytable
    INSERT INTO `{dst}` 
      SELECT * FROM `{dst}_tmp`
      WHERE date = '{replaceDay}'
    ;
COMMIT TRANSACTION;
"""
updateDef=dict(
    clusterBy="CLUSTER BY isValidBest, ServerContinent, ServerSite",
    tableOpts="OPTIONS (require_partition_filter = true)",
    topColumns="server.Geo.ContinentCode AS ServerContinent, server.Site AS ServerSite,",
    uuidCol="a.UUID"
)

def updatePartition(**argv):
    """updatePartition
        Required: src, dst, replaceDay
        Optional: tableOpts, clusterBy, topColumns, uuidCol
    """,
    av = updateDef.copy()
    av.update(argv)
    run_query(updateQ,**av)
def destroyTable(**argv):
    """Destroy and recreate a Table - EXTREMELY EXPENSIVE
        Avoid embedding this in anything automatic

        This deletes a table and recreates it with the single specified partition
        (Same arguments as updatePartition)
    """
    destroyQ="""
        CREATE OR REPLACE TABLE `{dst}` 
            PARTITION BY date
            {clusterBy}
            {tableOpts}
        AS (
            SELECT *, {topColumns}
            FROM `{src}`  -- ('{replaceDay}')  Function form
            WHERE date = '{replaceDay}'
                AND FARM_FINGERPRINT({uuidCol}) & 0xF =  0
        );
    
    """
    av = updateDef.copy()
    av.update(argv)
    run_query(destroyQ,**av)

# Unit Test
if True:
    testDest="mlab-collaboration.mm_preproduction.test_table"
    uuidCol="raw.Metadata.UUID"
    destroyTable(dst=testDest, src="measurement-lab.ndt.scamper1", replaceDay="2025-05-01", uuidCol=uuidCol, clusterBy='', topColumns='')
    updatePartition(dst=testDest, src="measurement-lab.ndt.scamper1", replaceDay="2025-05-02", uuidCol=uuidCol, clusterBy='', topColumns='')
    testQ = """
        SELECT MIN(date), MAX(date) FROM `{dst}` WHERE date > "2025-01-01"
    """.format(dst=testDest)
    display(run_query(testQ)) # Should be 2025-05-01	2025-05-02

AttributeError: 'DataFrame' object has no attribute 'display'

,f0_,f1_
0,2025-05-01,2025-05-02


In [35]:
print (numberedQuery)

1 
2         SELECT MIN(date), MAX(date) FROM `mlab-collaboration.mm_preproduction.test_table WHERE date > "2025-01-01"
3     


In [11]:
# Get the last partition in an arbitrary table
src="mlab-collaboration.mm_preproduction.extended_intermediate_downloads_DS16"

query="""
SELECT MAX(date) AS Ending FROM `{src}` WHERE date >= "2026-01-01"
"""
run_query(query, src=src)

,Ending
0,2026-04-24
